# Notebook 06 - Robustness: the Fail-only label variant

A robustness check on the central results. The main analysis defines at-risk as Fail or Withdrawn
(52.8 percent positive). Here at-risk is redefined as **Fail only**, with Withdrawn moved into the
negative class alongside Pass and Distinction. This drops the positive rate to about 21.6 percent,
turning a near-balanced problem into an imbalanced one and stressing three claims: that calibration
still holds, that discrimination survives, and above all that the false-positive-rate gap, the one
real equity finding, holds under a different label.

To stay directly comparable, this reuses the exact imputed features and the saved train/calibration
/test split from NB02; only the target changes. Models are retrained and recalibrated on the new
label, then the fairness audit is rerun. No SHAP or DiCE, so it is fast.

**Inputs:** `model_ready_week{w}` (features + split), `features_week{w}` (for `final_result`),
`models_week{w}.joblib` (feature list only), and the main `metrics_summary.csv` / `fairness_summary.csv`
for the comparison. **Outputs:** `results/robustness_metrics.csv`, `results/robustness_fairness.csv`,
`results/robustness_comparison.csv`, and figures in `results/figures/`.

## 0. Setup

In [1]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')

PROC   = ROOT / 'results' / 'processed'
MODELS = ROOT / 'results' / 'models'
FIG    = ROOT / 'results' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

CUTOFF_WEEKS = [5, 10, 15, 25]
SEED = 42
B_BOOT = 1000
THRESH = 0.5
KEYS = ['code_module', 'code_presentation', 'id_student']
MODEL_ORDER = ['logreg', 'rf', 'hgb']
DEPRIVED, AFFLUENT = [0, 1, 2], [7, 8, 9]

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (roc_auc_score, recall_score, f1_score,
                             precision_score, accuracy_score, brier_score_loss)

## 1. Helpers (shared with NB02/NB04)

In [3]:
def load(stem):
    for e in ('.parquet', '.csv'):
        if (Path(str(stem) + e)).exists():
            return pd.read_parquet(str(stem) + e) if e == '.parquet' else pd.read_csv(str(stem) + e)
    return None

def ece(y, p, n_bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0, 1, n_bins + 1)
    b = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    n = len(p); e = 0.0
    for k in range(n_bins):
        msk = b == k
        if msk.sum(): e += (msk.sum()/n) * abs(y[msk].mean() - p[msk].mean())
    return float(e)

def fit_calibrator(p_cal, y_cal):
    p_cal, y_cal = np.asarray(p_cal, float), np.asarray(y_cal, int)
    if len(p_cal) >= 30:
        iso = IsotonicRegression(out_of_bounds='clip'); iso.fit(p_cal, y_cal)
        return ('isotonic', iso)
    lr = LogisticRegression(); lr.fit(p_cal.reshape(-1, 1), y_cal)
    return ('platt', lr)

def apply_calibrator(cal, p):
    kind, model = cal; p = np.asarray(p, float)
    return model.predict(p) if kind == 'isotonic' else model.predict_proba(p.reshape(-1, 1))[:, 1]

def build_models():
    return {
        'logreg': Pipeline([('scaler', StandardScaler()),
                            ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED))]),
        'rf': RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=SEED),
        'hgb': HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, l2_regularization=1.0, random_state=SEED),
    }

def _rates(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    flag = pred.mean() if len(pred) else np.nan
    tpr = pred[y == 1].mean() if (y == 1).any() else np.nan
    fpr = pred[y == 0].mean() if (y == 0).any() else np.nan
    return flag, tpr, fpr

def fairness_gaps(yA, pA, yB, pB, B, seed):
    yA, pA, yB, pB = map(np.asarray, (yA, pA, yB, pB))
    fA, fB = _rates(yA, pA), _rates(yB, pB)
    raw = np.array(fA) - np.array(fB)
    r = np.random.default_rng(seed); nA, nB = len(yA), len(yB)
    boot = np.empty((B, 3))
    for i in range(B):
        ia, ib = r.integers(0, nA, nA), r.integers(0, nB, nB)
        boot[i] = np.array(_rates(yA[ia], pA[ia])) - np.array(_rates(yB[ib], pB[ib]))
    ci = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    return fA, fB, raw, ci

## 2. Retrain, recalibrate, and audit on the Fail-only label

In [4]:
metrics_rows, fair_rows = [], []
for w in CUTOFF_WEEKS:
    FEATURES = joblib.load(MODELS / f'models_week{w}.joblib')['features']
    ready = load(PROC / f'model_ready_week{w}')
    feat  = load(PROC / f'features_week{w}')[KEYS + ['final_result']]
    ready = ready.merge(feat, on=KEYS, how='left')

    y = (ready['final_result'] == 'Fail').astype(int)          # Fail-only label
    X = ready[FEATURES].astype(float)
    tr, ca, te = (ready['split'] == 'train'), (ready['split'] == 'calib'), (ready['split'] == 'test')

    Xtr, ytr = X[tr], y[tr]
    imd = pd.to_numeric(ready['imd_band_ord'], errors='coerce')

    for name, est in build_models().items():
        if name == 'hgb':
            est.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
        else:
            est.fit(Xtr, ytr)
        cal = fit_calibrator(est.predict_proba(X[ca])[:, 1], y[ca])
        p_un = est.predict_proba(X[te])[:, 1]
        p_cal = apply_calibrator(cal, p_un)
        yte = y[te].to_numpy(); pred = (p_cal >= THRESH).astype(int)
        metrics_rows.append({
            'cutoff_week': w, 'model': name, 'positive_rate': float(ytr.mean()),
            'auroc': roc_auc_score(yte, p_un), 'recall': recall_score(yte, pred, zero_division=0),
            'f1': f1_score(yte, pred, zero_division=0), 'brier_cal': brier_score_loss(yte, p_cal),
            'ece_uncal': ece(yte, p_un), 'ece_cal': ece(yte, p_cal),
        })

        imd_te = imd[te].to_numpy()
        groups = {
            'imd': (np.isin(imd_te, DEPRIVED), np.isin(imd_te, AFFLUENT)),
            'disability': ((ready['disability'][te] == 'Y').to_numpy(), (ready['disability'][te] == 'N').to_numpy()),
            'age': ((ready['age_band'][te] == '0-35').to_numpy(), (ready['age_band'][te] == '55<=').to_numpy()),
        }
        for attr, (mA, mB) in groups.items():
            if mA.sum() == 0 or mB.sum() == 0: continue
            fA, fB, raw, ci = fairness_gaps(yte[mA], pred[mA], yte[mB], pred[mB], B_BOOT, SEED)
            fair_rows.append({
                'cutoff_week': w, 'model': name, 'attribute': attr,
                'dp_gap': raw[0], 'eo_gap': raw[1], 'eo_lo': ci[0,1], 'eo_hi': ci[1,1],
                'fpr_gap': raw[2], 'fpr_lo': ci[0,2], 'fpr_hi': ci[1,2],
            })
    print(f'week {w:2d}: Fail-only retrain + audit done (positive rate {ytr.mean():.3f})')

rob_metrics = pd.DataFrame(metrics_rows)
rob_fair = pd.DataFrame(fair_rows)
rob_metrics.to_csv(ROOT / 'results' / 'robustness_metrics.csv', index=False)
rob_fair.to_csv(ROOT / 'results' / 'robustness_fairness.csv', index=False)
print('\nFail-only metrics:')
print(rob_metrics[['cutoff_week','model','auroc','recall','ece_cal']].round(4).to_string(index=False))

week  5: Fail-only retrain + audit done (positive rate 0.215)
week 10: Fail-only retrain + audit done (positive rate 0.215)
week 15: Fail-only retrain + audit done (positive rate 0.215)
week 25: Fail-only retrain + audit done (positive rate 0.215)

Fail-only metrics:
 cutoff_week  model  auroc  recall  ece_cal
           5 logreg 0.6820  0.0000   0.0063
           5     rf 0.6823  0.0028   0.0088
           5    hgb 0.7065  0.0330   0.0081
          10 logreg 0.7097  0.0014   0.0131
          10     rf 0.7209  0.0035   0.0134
          10    hgb 0.7442  0.0681   0.0079
          15 logreg 0.7312  0.0021   0.0094
          15     rf 0.7450  0.0098   0.0163
          15    hgb 0.7694  0.1179   0.0065
          25 logreg 0.7704  0.0021   0.0127
          25     rf 0.7905  0.3053   0.0138
          25    hgb 0.8126  0.2730   0.0128


## 3. Side-by-side comparison with the main results

In [7]:
import os
os.chdir('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
print('metrics_summary:', os.path.exists('results/metrics_summary.csv'))
print('fairness_summary:', os.path.exists('results/fairness_summary.csv'))
print('in results/:', sorted(f for f in os.listdir('results') if f.endswith('.csv')))

metrics_summary: True
fairness_summary: True
in results/: ['agreement_summary.csv', 'fairness_summary.csv', 'faithfulness_adjusted_summary.csv', 'faithfulness_summary.csv', 'metrics_summary.csv', 'recourse_gaps.csv', 'robustness_fairness.csv', 'robustness_metrics.csv', 'trust_equity_table.csv']


In [8]:
main_m = load('results/metrics_summary')
main_f = load('results/fairness_summary')
rows = []
for w in CUTOFF_WEEKS:
    for model in MODEL_ORDER:
        mm = main_m[(main_m.cutoff_week==w) & (main_m.model==model)]
        rm = rob_metrics[(rob_metrics.cutoff_week==w) & (rob_metrics.model==model)]
        mf = main_f[(main_f.cutoff_week==w) & (main_f.model==model) & (main_f.attribute=='imd')]
        rf = rob_fair[(rob_fair.cutoff_week==w) & (rob_fair.model==model) & (rob_fair.attribute=='imd')]
        if len(mm) and len(rm):
            rows.append({
                'cutoff_week': w, 'model': model,
                'auroc_main': float(mm.auroc.iloc[0]), 'auroc_failonly': float(rm.auroc.iloc[0]),
                'ece_main': float(mm.ece_cal.iloc[0]), 'ece_failonly': float(rm.ece_cal.iloc[0]),
                'fpr_gap_imd_main': float(mf.fpr_gap.iloc[0]) if len(mf) else np.nan,
                'fpr_gap_imd_failonly': float(rf.fpr_gap.iloc[0]) if len(rf) else np.nan,
                'fpr_failonly_lo': float(rf.fpr_lo.iloc[0]) if len(rf) else np.nan,
                'fpr_failonly_hi': float(rf.fpr_hi.iloc[0]) if len(rf) else np.nan,
            })
comp = pd.DataFrame(rows)
comp.to_csv(ROOT / 'results' / 'robustness_comparison.csv', index=False)
print('Main vs Fail-only (deprivation FPR gap is the key comparison):')
print(comp.round(4).to_string(index=False))

# does the deprivation FPR finding survive? (hgb)
h = comp[comp.model=='hgb']
survive = [(int(r.cutoff_week)) for _, r in h.iterrows() if r.fpr_failonly_lo > 0]
print('\nDeprivation FPR gap still excludes zero under Fail-only (hgb) at:',
      survive if survive else 'no cutoff')

Main vs Fail-only (deprivation FPR gap is the key comparison):
 cutoff_week  model  auroc_main  auroc_failonly  ece_main  ece_failonly  fpr_gap_imd_main  fpr_gap_imd_failonly  fpr_failonly_lo  fpr_failonly_hi
           5 logreg      0.8297          0.6820    0.0220        0.0063            0.0912                0.0004          -0.0025           0.0033
           5     rf      0.8463          0.6823    0.0253        0.0088            0.0477               -0.0024          -0.0061           0.0005
           5    hgb      0.8527          0.7065    0.0187        0.0081            0.0462                0.0074           0.0001           0.0146
          10 logreg      0.8937          0.7097    0.0142        0.0131            0.0713                0.0031          -0.0028           0.0094
          10     rf      0.9020          0.7209    0.0165        0.0134            0.0375                0.0000           0.0000           0.0000
          10    hgb      0.9050          0.7442    0.0191    

## 4. Figures

In [9]:
h_m = main_m[main_m.model=='hgb'].sort_values('cutoff_week')
h_r = rob_metrics[rob_metrics.model=='hgb'].sort_values('cutoff_week')
cut = h_m['cutoff_week'].to_numpy()

plt.figure(figsize=(6,4))
plt.plot(cut, h_m['auroc'], marker='o', label='main (Fail+Withdrawn)')
plt.plot(cut, h_r['auroc'], marker='s', label='Fail-only')
plt.xticks(cut); plt.xlabel('week cutoff'); plt.ylabel('test AUROC')
plt.title('Discrimination under label change (gradient boosting)')
plt.legend(frameon=False); plt.tight_layout(); plt.savefig(FIG/'fig_robust_auroc.png', dpi=200); plt.close()

plt.figure(figsize=(6,4))
plt.plot(cut, h_m['ece_cal'], marker='o', label='main')
plt.plot(cut, h_r['ece_cal'], marker='s', label='Fail-only')
plt.xticks(cut); plt.xlabel('week cutoff'); plt.ylabel('calibrated ECE')
plt.title('Calibration under label change (gradient boosting)')
plt.legend(frameon=False); plt.tight_layout(); plt.savefig(FIG/'fig_robust_ece.png', dpi=200); plt.close()

mf = main_f[(main_f.model=='hgb') & (main_f.attribute=='imd')].sort_values('cutoff_week')
rf = rob_fair[(rob_fair.model=='hgb') & (rob_fair.attribute=='imd')].sort_values('cutoff_week')
x = np.arange(len(cut)); wd = 0.38
plt.figure(figsize=(6.2,4))
plt.bar(x-wd/2, mf['fpr_gap'], wd, label='main', color='#888780')
plt.bar(x+wd/2, rf['fpr_gap'], wd, label='Fail-only', color='#993C1D',
        yerr=[rf['fpr_gap']-rf['fpr_lo'], rf['fpr_hi']-rf['fpr_gap']], capsize=3)
plt.axhline(0, color='#444441', lw=0.8); plt.xticks(x, [f'wk {int(c)}' for c in cut])
plt.ylabel('deprivation FPR gap'); plt.title('Does the FPR finding survive the label change?')
plt.legend(frameon=False); plt.tight_layout(); plt.savefig(FIG/'fig_robust_fpr_gap.png', dpi=200); plt.close()
print('saved 3 robustness figures to', FIG)

saved 3 robustness figures to /content/drive/MyDrive/StudentEWS_Research/student-ews-research/results/figures


## How to read this

`robustness_comparison.csv` puts AUROC, calibrated ECE, and the deprivation FPR gap side by side for
the main (Fail or Withdrawn) and Fail-only labels. Three questions:

* **Discrimination:** does AUROC hold under the harder, imbalanced label? A modest drop is expected and fine.
* **Calibration:** does ECE stay low? At about 22 percent positive the calibration step is doing more work,
  so this is where the hybrid calibration earns its keep.
* **The headline:** does the deprivation FPR gap still exclude zero? If it survives the label change, the
  over-flagging of disadvantaged students is robust to how at-risk is defined, which strengthens the C4/C6
  claim. If it vanishes, the finding is label-specific and must be reported as such.

This is the last analysis before writing the paper. With it, the main findings have a robustness section.